# Task 16: XGBoost vs TabPFN v2 — gene-disjoint 8:1:1 holdout

在相同 64D 特征上比较 XGBoost、TabPFN 和 rank ensemble。按 `Gene` 分组切分约 80% train、10% validation、10% test；同一 gene 不跨 split。所有预处理只在 train 上拟合，validation 只选择 ensemble weight，test 只用于最终评估。

由于 gene 大小不同，保持 gene 完整时无法保证样本数恰好为 80:10:10。本 notebook 从多个候选 group splits 中选择样本比例和正例率最接近目标的切分，并打印实际比例。

In [1]:
from pathlib import Path
from collections import OrderedDict
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

DEVICE = "cuda"
MODEL_PATH = Path("/mnt/volume6/czj/labLGN/LabLZ/models/tabpfn-v2-classifier.ckpt")
BASE = Path("/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial")
FEATURES_CSV = BASE / "features_v3.csv"
AM_CSV = BASE / "alphamissense_completed.csv"
TMD_CSV = BASE / "tmd_features.csv"
OOF_CSV = BASE / "task16_holdout_predictions.csv"
METRICS_CSV = BASE / "task16_holdout_metrics.csv"
SPLIT_CSV = BASE / "task16_holdout_split.csv"

RANDOM_STATE = 42
N_SPLIT_CANDIDATES = 2000
N_COMPONENTS = 50
STRUCT_COLS = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand", "ss_coil", "delta_hydrophobicity"]
DDG_COLS = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
TMD_COLS = ["in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]

assert MODEL_PATH.exists(), f"Missing local TabPFN checkpoint: {MODEL_PATH}"

df = pd.read_csv(FEATURES_CSV)
required = ["KEY", "Gene", "Mutation_used", "source", "Mislocalized"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns in features_v3.csv: {missing}"
assert len(df) == 2179 and df["KEY"].is_unique
assert df["Mislocalized"].isin([0, 1]).all() and int(df["Mislocalized"].sum()) == 236

esm_cols = [c for c in df.columns if c.startswith("esm_")]
assert len(esm_cols) == 1280, f"Expected 1280 ESM columns, found {len(esm_cols)}"
assert all(c in df.columns for c in STRUCT_COLS)

for name in DDG_COLS:
    table = pd.read_csv(BASE / f"{name}.csv")
    assert table["KEY"].is_unique and name in table.columns
    assert not (set(table["KEY"]) - set(df["KEY"]))
    df = df.merge(table[["KEY", name]], on="KEY", how="left", validate="one_to_one")

tmd = pd.read_csv(TMD_CSV)
assert tmd["KEY"].is_unique and all(c in tmd.columns for c in TMD_COLS)
assert set(tmd["KEY"]) == set(df["KEY"])
df = df.merge(tmd[["KEY"] + TMD_COLS], on="KEY", how="left", validate="one_to_one")

am = pd.read_csv(AM_CSV)
am_required = ["KEY", "final_alphamissense_score", "alphamissense_status"]
assert am["KEY"].is_unique and all(c in am.columns for c in am_required)
assert set(am["KEY"]) == set(df["KEY"])
df = df.merge(am[am_required], on="KEY", how="left", validate="one_to_one")

assert len(df) == 2179 and df["KEY"].is_unique
y = df["Mislocalized"].astype(int).to_numpy()
groups = df["Gene"].astype(str).to_numpy()
print(f"n={len(df)}, genes={df['Gene'].nunique()}, positives={y.sum()}, prevalence={y.mean():.4f}")
print(f"AlphaMissense available: {df['final_alphamissense_score'].notna().sum()}/{len(df)}")

n=2179, genes=871, positives=236, prevalence=0.1083
AlphaMissense available: 2140/2179


In [2]:
def split_score(train_idx, val_idx, test_idx):
    target_sizes = np.array([0.8, 0.1, 0.1])
    actual_sizes = np.array([len(train_idx), len(val_idx), len(test_idx)]) / len(y)
    size_error = np.abs(actual_sizes - target_sizes).sum()
    overall_rate = y.mean()
    rates = np.array([y[train_idx].mean(), y[val_idx].mean(), y[test_idx].mean()])
    prevalence_error = np.abs(rates - overall_rate).sum()
    class_penalty = 100 if any(y[idx].sum() == 0 for idx in [train_idx, val_idx, test_idx]) else 0
    return size_error + prevalence_error + class_penalty

def choose_grouped_811_split():
    best = None
    outer = GroupShuffleSplit(n_splits=N_SPLIT_CANDIDATES, test_size=0.20, random_state=RANDOM_STATE)
    for candidate, (train_idx, temp_idx) in enumerate(outer.split(df, y, groups)):
        temp_groups = groups[temp_idx]
        inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE + candidate)
        val_local, test_local = next(inner.split(temp_idx, y[temp_idx], temp_groups))
        val_idx = temp_idx[val_local]
        test_idx = temp_idx[test_local]
        score = split_score(train_idx, val_idx, test_idx)
        if best is None or score < best[0]:
            best = (score, train_idx, val_idx, test_idx)
    return best[1], best[2], best[3]

train_idx, val_idx, test_idx = choose_grouped_811_split()
split_indices = {"train": train_idx, "validation": val_idx, "test": test_idx}

gene_sets = {name: set(groups[idx]) for name, idx in split_indices.items()}
assert gene_sets["train"].isdisjoint(gene_sets["validation"])
assert gene_sets["train"].isdisjoint(gene_sets["test"])
assert gene_sets["validation"].isdisjoint(gene_sets["test"])
assert set(np.concatenate(list(split_indices.values()))) == set(range(len(df)))

split_label = np.full(len(df), "", dtype=object)
for name, idx in split_indices.items():
    split_label[idx] = name
    print(f"{name:10s}: n={len(idx):4d} ({len(idx)/len(df):.3f}), genes={len(gene_sets[name]):3d}, positives={y[idx].sum():3d}, prevalence={y[idx].mean():.4f}")

split_out = df[["KEY", "Gene", "Mutation_used", "Mislocalized"]].copy()
split_out["split"] = split_label
split_out.to_csv(SPLIT_CSV, index=False)

train     : n=1748 (0.802), genes=696, positives=189, prevalence=0.1081
validation: n= 221 (0.101), genes= 87, positives= 24, prevalence=0.1086
test      : n= 210 (0.096), genes= 88, positives= 23, prevalence=0.1095


In [3]:
X_esm = df[esm_cols].to_numpy(dtype=np.float32)
X_struct = df[STRUCT_COLS].to_numpy(dtype=np.float32)
X_ddg = df[DDG_COLS].to_numpy(dtype=np.float32)
X_tmd = df[TMD_COLS].to_numpy(dtype=np.float32)

def prepare_64d(train_idx, val_idx, test_idx):
    esm_imputer = SimpleImputer(strategy="median")
    esm_scaler = StandardScaler()
    esm_train = esm_scaler.fit_transform(esm_imputer.fit_transform(X_esm[train_idx]))
    esm_val = esm_scaler.transform(esm_imputer.transform(X_esm[val_idx]))
    esm_test = esm_scaler.transform(esm_imputer.transform(X_esm[test_idx]))
    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    pc_train = pca.fit_transform(esm_train)
    pc_val = pca.transform(esm_val)
    pc_test = pca.transform(esm_test)

    struct_imputer = SimpleImputer(strategy="median")
    struct_scaler = StandardScaler()
    struct_train = struct_scaler.fit_transform(struct_imputer.fit_transform(X_struct[train_idx]))
    struct_val = struct_scaler.transform(struct_imputer.transform(X_struct[val_idx]))
    struct_test = struct_scaler.transform(struct_imputer.transform(X_struct[test_idx]))

    extra_imputer = SimpleImputer(strategy="median")
    extra_train_raw = np.hstack([X_ddg[train_idx], X_tmd[train_idx]])
    extra_val_raw = np.hstack([X_ddg[val_idx], X_tmd[val_idx]])
    extra_test_raw = np.hstack([X_ddg[test_idx], X_tmd[test_idx]])
    extra_train = extra_imputer.fit_transform(extra_train_raw)
    extra_val = extra_imputer.transform(extra_val_raw)
    extra_test = extra_imputer.transform(extra_test_raw)

    train = np.hstack([pc_train, struct_train, extra_train]).astype(np.float32)
    val = np.hstack([pc_val, struct_val, extra_val]).astype(np.float32)
    test = np.hstack([pc_test, struct_test, extra_test]).astype(np.float32)
    assert train.shape[1] == val.shape[1] == test.shape[1] == 64
    assert np.isfinite(train).all() and np.isfinite(val).all() and np.isfinite(test).all()
    return train, val, test

X_train, X_val, X_test = prepare_64d(train_idx, val_idx, test_idx)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print(f"Prepared matrices: train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}")

Prepared matrices: train=(1748, 64), validation=(221, 64), test=(210, 64)


In [4]:
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.5,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)
xgb.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
xgb_val = xgb.predict_proba(X_val)[:, 1]
xgb_test = xgb.predict_proba(X_test)[:, 1]

from tabpfn import TabPFNClassifier
tabpfn = TabPFNClassifier(device=DEVICE, model_path=str(MODEL_PATH))
tabpfn.fit(X_train, y_train)
tab_val = tabpfn.predict_proba(X_val)[:, 1]
tab_test = tabpfn.predict_proba(X_test)[:, 1]

def percentile_rank(reference, values):
    reference = np.asarray(reference)
    values = np.asarray(values)
    sorted_reference = np.sort(reference)
    return np.searchsorted(sorted_reference, values, side="right") / len(sorted_reference)

xgb_val_rank = percentile_rank(xgb_val, xgb_val)
tab_val_rank = percentile_rank(tab_val, tab_val)
weights = np.linspace(0.0, 1.0, 21)
validation_results = []
for tab_weight in weights:
    ensemble_val = (1 - tab_weight) * xgb_val_rank + tab_weight * tab_val_rank
    validation_results.append({
        "tabpfn_weight": tab_weight,
        "auroc": roc_auc_score(y_val, ensemble_val),
        "auprc": average_precision_score(y_val, ensemble_val),
    })
validation_results = pd.DataFrame(validation_results)
best_row = validation_results.sort_values(["auprc", "auroc"], ascending=False).iloc[0]
best_tab_weight = float(best_row["tabpfn_weight"])
print(f"Selected TabPFN ensemble weight on validation: {best_tab_weight:.2f}")

xgb_test_rank = percentile_rank(xgb_val, xgb_test)
tab_test_rank = percentile_rank(tab_val, tab_test)
ensemble_test = (1 - best_tab_weight) * xgb_test_rank + best_tab_weight * tab_test_rank

Selected TabPFN ensemble weight on validation: 0.60


In [5]:
def metric_record(scope, model, truth, prediction):
    return {
        "scope": scope,
        "model": model,
        "n": len(truth),
        "positives": int(np.sum(truth)),
        "prevalence": float(np.mean(truth)),
        "auroc": roc_auc_score(truth, prediction),
        "auprc": average_precision_score(truth, prediction),
    }

test_predictions = OrderedDict([
    ("xgboost_64", xgb_test),
    ("tabpfn_64", tab_test),
    ("ensemble_64", ensemble_test),
])
records = []
print("\nFinal held-out test results")
print(f"n={len(test_idx)}, positives={y_test.sum()}, prevalence={y_test.mean():.4f}")
for model, prediction in test_predictions.items():
    record = metric_record("full_test", model, y_test, prediction)
    records.append(record)
    print(f"  {model:12s}: AUROC={record['auroc']:.4f}, AUPRC={record['auprc']:.4f}")

am_test = df.iloc[test_idx]["final_alphamissense_score"].to_numpy(dtype=float)
paired = np.isfinite(am_test)
paired_y = y_test[paired]
am_record = metric_record("paired_alphamissense_test", "alphamissense", paired_y, am_test[paired])
records.append(am_record)
print("\nPaired AlphaMissense test comparison")
print(f"n={paired.sum()}, positives={paired_y.sum()}, prevalence={paired_y.mean():.4f}")
print(f"  {'alphamissense':12s}: AUROC={am_record['auroc']:.4f}, AUPRC={am_record['auprc']:.4f}")
for model, prediction in test_predictions.items():
    record = metric_record("paired_alphamissense_test", model, paired_y, prediction[paired])
    records.append(record)
    print(f"  {model:12s}: AUROC={record['auroc']:.4f}, AUPRC={record['auprc']:.4f}")

prediction_out = df.iloc[test_idx][["KEY", "Gene", "Mutation_used", "source", "Mislocalized", "final_alphamissense_score", "alphamissense_status"]].copy()
prediction_out["xgboost_64"] = xgb_test
prediction_out["tabpfn_64"] = tab_test
prediction_out["ensemble_64"] = ensemble_test
prediction_out["selected_tabpfn_weight"] = best_tab_weight
prediction_out.to_csv(OOF_CSV, index=False)
pd.DataFrame(records).to_csv(METRICS_CSV, index=False)
print(f"\nSaved: {OOF_CSV}")
print(f"Saved: {METRICS_CSV}")
print(f"Saved: {SPLIT_CSV}")


Final held-out test results
n=210, positives=23, prevalence=0.1095
  xgboost_64  : AUROC=0.6266, AUPRC=0.2075
  tabpfn_64   : AUROC=0.6002, AUPRC=0.2441
  ensemble_64 : AUROC=0.6223, AUPRC=0.2056

Paired AlphaMissense test comparison
n=204, positives=22, prevalence=0.1078
  alphamissense: AUROC=0.6190, AUPRC=0.1552
  xgboost_64  : AUROC=0.6391, AUPRC=0.2113
  tabpfn_64   : AUROC=0.6155, AUPRC=0.2500
  ensemble_64 : AUROC=0.6380, AUPRC=0.2096

Saved: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task16_holdout_predictions.csv
Saved: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task16_holdout_metrics.csv
Saved: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task16_holdout_split.csv


## Interpretation boundary

最终模型比较只读取 held-out test。Validation 指标仅用于选择 ensemble weight，不能作为最终性能。该单次 8:1:1 holdout 的 test set 约 218 行，方差会明显高于 5-fold pooled OOF；模型差异需要结合 confidence interval 或后续重复 grouped holdout 判断。